# Taxi Destination Dataset Analysis

Decision-focused notebook for deriving preprocessing rules used in the model notebooks. It analyzes raw data validity, temporal feature support, prefix generation, spatial filtering, duration artifacts, and GPS scaling.

## Preprocessing Rules Summary

Rules supported by this notebook:

- Drop train rows with `MISSING_DATA=True`, invalid polylines, or trajectories with fewer than 2 GPS points.
- Generate synthetic train/validation prefixes by sampling prefix lengths from the unlabeled competition `test.csv` prefix-length distribution.
- Apply broad train-only spatial sanity bounds to start and final destination points: longitude `[-8.85, -7.40]`, latitude `[40.85, 41.65]`.
- Drop train-only duration artifacts where full trip duration is greater than 240 minutes and the trajectory never goes more than 20 km from its start.
- Extract fixed `5 first + 5 last` GPS points in `code/build_preprocessed_datasets.py` and standardize GPS columns with train-fitted statistics.

`solutions.csv` is intentionally not used.

## Setup

In [ ]:
from pathlib import Path
import ast
import random

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd

SEED = 42
PREFIX_SAMPLE_FACTOR = 1
PREFIX_LENGTH_PERCENTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
RUN_GPS_DENSITY = False

SECONDS_PER_GPS_STEP = 15
BROAD_SANITY_BOUNDS = {"lon_min": -8.85, "lon_max": -7.40, "lat_min": 40.85, "lat_max": 41.65}
PORTO_CORE_BOUNDS = {"lon_min": -8.75, "lon_max": -8.45, "lat_min": 41.05, "lat_max": 41.30}
TEST_ENVELOPE_MARGIN = 0.02
DURATION_ARTIFACT_MINUTES = 240
DURATION_ARTIFACT_MAX_DISTANCE_FROM_START_KM = 20

random.seed(SEED)
np.random.seed(SEED)

TRAIN_PATH = Path("/kaggle/input/datasets/louishogge/taxi-dataset/data/train.csv")
TEST_PATH = Path("/kaggle/input/datasets/louishogge/taxi-dataset/data/test.csv")
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)

In [ ]:
def parse_polyline(polyline):
    """Parse and validate a POLYLINE value as [longitude, latitude] points."""
    if isinstance(polyline, list):
        parsed = polyline
    else:
        parsed = ast.literal_eval(polyline)
    if not isinstance(parsed, list):
        raise ValueError(f"POLYLINE must parse to a list, got {type(parsed)!r}")
    for point in parsed:
        if not isinstance(point, (list, tuple)) or len(point) != 2:
            raise ValueError(f"Invalid GPS point in POLYLINE: {point!r}")
    return parsed


def safe_parse_polyline(polyline):
    """Parse a POLYLINE value, returning None for malformed rows."""
    try:
        return parse_polyline(polyline)
    except Exception:
        return None


def add_polyline_profile(data, require_target=False):
    """Attach parsed trajectories, point counts, and validity flags."""
    profiled = data.copy()
    parsed = profiled["POLYLINE"].map(safe_parse_polyline)
    profiled["POLYLINE_POINTS"] = parsed
    profiled["POLYLINE_LENGTH"] = parsed.map(lambda points: np.nan if points is None else len(points))
    min_length = 2 if require_target else 1
    profiled["VALID_POLYLINE"] = profiled["POLYLINE_LENGTH"] >= min_length
    return profiled


def first_point(points):
    """Return the first GPS point, or NaNs for invalid input."""
    return points[0] if isinstance(points, list) and len(points) else [np.nan, np.nan]


def last_point(points):
    """Return the last GPS point, or NaNs for invalid input."""
    return points[-1] if isinstance(points, list) and len(points) else [np.nan, np.nan]


def point_frame_from_rows(profiled, label, point_getter):
    """Build a coordinate table for one representative point per trip."""
    rows = []
    valid = profiled.loc[profiled["VALID_POLYLINE"]]
    for row_id, row in valid.iterrows():
        point = point_getter(row["POLYLINE_POINTS"])
        rows.append({
            "SOURCE_ROW_ID": row_id,
            "TRIP_ID": row["TRIP_ID"] if "TRIP_ID" in row.index else row_id,
            "LONGITUDE": float(point[0]),
            "LATITUDE": float(point[1]),
            "POINT_TYPE": label,
        })
    return pd.DataFrame(rows)


def flatten_profiled_points(profiled, label):
    """Flatten valid trajectories into a point-level coordinate table."""
    rows = []
    valid = profiled.loc[profiled["VALID_POLYLINE"]]
    for row_id, row in valid.iterrows():
        trip_id = row["TRIP_ID"] if "TRIP_ID" in row.index else row_id
        for point_index, point in enumerate(row["POLYLINE_POINTS"]):
            rows.append({
                "SOURCE_ROW_ID": row_id,
                "TRIP_ID": trip_id,
                "POINT_INDEX": point_index,
                "LONGITUDE": float(point[0]),
                "LATITUDE": float(point[1]),
                "POINT_TYPE": label,
            })
    return pd.DataFrame(rows)


def bounds_mask(points, bounds):
    """Return True for points inside a longitude/latitude bounds dictionary."""
    return (
        points["LONGITUDE"].between(bounds["lon_min"], bounds["lon_max"], inclusive="both")
        & points["LATITUDE"].between(bounds["lat_min"], bounds["lat_max"], inclusive="both")
    )


def compute_envelope_bounds(points, margin=TEST_ENVELOPE_MARGIN):
    """Compute a padded coordinate envelope around observed points."""
    return {
        "lon_min": float(points["LONGITUDE"].min() - margin),
        "lon_max": float(points["LONGITUDE"].max() + margin),
        "lat_min": float(points["LATITUDE"].min() - margin),
        "lat_max": float(points["LATITUDE"].max() + margin),
    }


def describe_lengths(**series_by_name):
    """Summarize trajectory or prefix length distributions."""
    summaries = []
    for name, values in series_by_name.items():
        summary = pd.Series(values).dropna().describe(percentiles=PREFIX_LENGTH_PERCENTILES)
        summary.name = name
        summaries.append(summary)
    return pd.concat(summaries, axis=1)


def sample_uniform_prefix_lengths(full_lengths, seed=SEED):
    """Sample one random feasible prefix length per full trajectory."""
    rng = np.random.default_rng(seed)
    values = []
    for full_length in np.asarray(full_lengths, dtype=int):
        if full_length >= 2:
            values.append(int(rng.integers(1, full_length)))
    return pd.Series(values, name="uniform_prefix")


def sample_prefix_lengths_like_test(full_lengths, test_prefix_lengths, seed=SEED, sample_factor=1):
    """Sample feasible train prefix lengths from the competition test distribution."""
    rng = np.random.default_rng(seed)
    full_lengths = np.asarray(full_lengths, dtype=int)
    test_prefix_lengths = np.asarray(test_prefix_lengths, dtype=int)
    full_lengths = full_lengths[full_lengths >= 2]
    test_prefix_lengths = test_prefix_lengths[test_prefix_lengths >= 1]

    sorted_positions = np.argsort(full_lengths)
    sorted_lengths = full_lengths[sorted_positions]
    sampled_lengths = []

    for _ in range(len(full_lengths) * sample_factor):
        desired_prefix_length = int(rng.choice(test_prefix_lengths))
        first_feasible = np.searchsorted(sorted_lengths, desired_prefix_length + 1, side="left")
        if first_feasible < len(sorted_positions):
            sampled_lengths.append(desired_prefix_length)
        else:
            sampled_lengths.append(int(max(full_lengths.max() - 1, 1)))
    return pd.Series(sampled_lengths, name="test_matched_prefix")


def max_distance_from_start_km(points):
    """Return the farthest observed point from the trip start, in kilometers."""
    if not isinstance(points, list) or len(points) == 0:
        return np.nan

    start_longitude, start_latitude = points[0]
    longitudes = np.radians(np.asarray([point[0] for point in points], dtype=float))
    latitudes = np.radians(np.asarray([point[1] for point in points], dtype=float))
    start_longitude = np.radians(float(start_longitude))
    start_latitude = np.radians(float(start_latitude))

    delta_longitude = longitudes - start_longitude
    delta_latitude = latitudes - start_latitude
    haversine_a = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(start_latitude) * np.cos(latitudes) * np.sin(delta_longitude / 2) ** 2
    )
    distances_km = 2 * 6371 * np.arctan2(np.sqrt(haversine_a), np.sqrt(1 - haversine_a))
    return float(np.nanmax(distances_km))


def add_bounds_rectangle(axis, bounds, label, color):
    """Draw a labeled longitude/latitude bounds rectangle."""
    width = bounds["lon_max"] - bounds["lon_min"]
    height = bounds["lat_max"] - bounds["lat_min"]
    rectangle = Rectangle((bounds["lon_min"], bounds["lat_min"]), width, height, fill=False, linewidth=2, edgecolor=color, label=label)
    axis.add_patch(rectangle)
    axis.text(bounds["lon_min"], bounds["lat_max"], label, color=color, fontsize=9, verticalalignment="bottom")

## Load Data

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

load_summary = pd.DataFrame({
    "dataset": ["train_full", "competition_test"],
    "rows": [len(train_raw), len(test_raw)],
    "columns": [train_raw.shape[1], test_raw.shape[1]],
})
display(load_summary)

## Compact Data Quality Profile

Summarize only raw fields that affect preprocessing decisions.

In [ ]:
PROFILE_COLUMNS = [
    "TRIP_ID",
    "CALL_TYPE",
    "ORIGIN_CALL",
    "ORIGIN_STAND",
    "TAXI_ID",
    "TIMESTAMP",
    "DAY_TYPE",
    "MISSING_DATA",
    "POLYLINE",
]

profile_rows = []
for dataset_name, data in [("train_full", train_raw), ("competition_test", test_raw)]:
    for column in PROFILE_COLUMNS:
        profile_rows.append({
            "dataset": dataset_name,
            "column": column,
            "null_pct": data[column].isna().mean() * 100,
            "n_unique": int(data[column].nunique(dropna=True)),
        })
compact_profile = pd.DataFrame(profile_rows)
display(compact_profile)

key_summary = pd.DataFrame({
    "dataset": ["train_full", "competition_test"],
    "rows": [len(train_raw), len(test_raw)],
    "duplicate_trip_ids": [int(train_raw["TRIP_ID"].duplicated().sum()), int(test_raw["TRIP_ID"].duplicated().sum())],
})
display(key_summary)

for column in ["CALL_TYPE", "DAY_TYPE", "MISSING_DATA"]:
    display(pd.concat([
        train_raw[column].value_counts(dropna=False).rename("train_full"),
        test_raw[column].value_counts(dropna=False).rename("competition_test"),
    ], axis=1).fillna(0).astype(int).rename_axis(column))

## Temporal Feature Checks

Check timestamp validity and local-time distributions. These plots support local cyclic features for ISO week, weekday, hour, and quarter-hour; they do not imply date-based filtering.

In [ ]:
PORTO_TIMEZONE = "Europe/Lisbon"


def build_time_profile(data, timezone=PORTO_TIMEZONE):
    """Create Porto-local timestamp features used for temporal diagnostics."""
    timestamps_utc = pd.to_datetime(data["TIMESTAMP"], unit="s", utc=True, errors="coerce")
    timestamps_local = timestamps_utc.dt.tz_convert(timezone)
    return pd.DataFrame({
        "TIMESTAMP_UTC": timestamps_utc,
        "TIMESTAMP_LOCAL": timestamps_local,
        "ISO_WEEK_LOCAL": timestamps_local.dt.isocalendar().week,
        "WEEKDAY_LOCAL": timestamps_local.dt.weekday,
        "HOUR_LOCAL": timestamps_local.dt.hour,
        "QUARTER_LOCAL": timestamps_local.dt.hour * 4 + timestamps_local.dt.minute // 15,
    }, index=data.index)


def summarize_time_profile(time_profile, name):
    """Summarize timestamp validity and local-time coverage."""
    valid = time_profile.dropna(subset=["TIMESTAMP_UTC"])
    return {
        "dataset": name,
        "rows": len(time_profile),
        "invalid_timestamps": int(time_profile["TIMESTAMP_UTC"].isna().sum()),
        "min_local": valid["TIMESTAMP_LOCAL"].min(),
        "max_local": valid["TIMESTAMP_LOCAL"].max(),
        "n_local_weeks": int(valid["ISO_WEEK_LOCAL"].nunique()),
    }


def percentage_share(values, categories):
    """Return category shares on a fixed category index."""
    values = pd.Series(values).dropna()
    return values.value_counts(normalize=True).reindex(categories, fill_value=0) * 100


def plot_percentage_comparison(train_values, test_values, categories, axis, title, xlabel, kind="bar"):
    """Plot train/test category shares on a shared percentage scale."""
    train_share = percentage_share(train_values, categories)
    test_share = percentage_share(test_values, categories)
    x = np.arange(len(categories))
    if kind == "bar":
        axis.bar(x - 0.2, train_share.to_numpy(), width=0.4, alpha=0.55, label="train")
        axis.bar(x + 0.2, test_share.to_numpy(), width=0.4, alpha=0.55, label="competition test")
        axis.set_xticks(x)
        axis.set_xticklabels(categories)
    else:
        axis.plot(categories, train_share.to_numpy(), marker="o", linewidth=1.5, label="train")
        axis.plot(categories, test_share.to_numpy(), marker="o", linewidth=1.5, label="competition test")
    axis.set_title(title)
    axis.set_xlabel(xlabel)
    axis.set_ylabel("Rows (%)")
    axis.legend()


train_temporal_raw = train_raw[["TIMESTAMP"]].copy()
test_temporal_raw = test_raw[["TIMESTAMP"]].copy()

train_time_profile = build_time_profile(train_temporal_raw)
test_time_profile = build_time_profile(test_temporal_raw)

display(pd.DataFrame([
    summarize_time_profile(train_time_profile, "train_full"),
    summarize_time_profile(test_time_profile, "competition_test"),
]))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_percentage_comparison(
    train_time_profile["ISO_WEEK_LOCAL"],
    test_time_profile["ISO_WEEK_LOCAL"],
    list(range(1, 54)),
    axes[0, 0],
    "ISO week distribution, Porto local time",
    "ISO week",
    kind="line",
)
plot_percentage_comparison(
    train_time_profile["WEEKDAY_LOCAL"],
    test_time_profile["WEEKDAY_LOCAL"],
    list(range(7)),
    axes[0, 1],
    "Weekday distribution, Porto local time",
    "Weekday (0=Monday)",
)
plot_percentage_comparison(
    train_time_profile["HOUR_LOCAL"],
    test_time_profile["HOUR_LOCAL"],
    list(range(24)),
    axes[1, 0],
    "Hour distribution, Porto local time",
    "Hour",
    kind="line",
)
plot_percentage_comparison(
    train_time_profile["QUARTER_LOCAL"],
    test_time_profile["QUARTER_LOCAL"],
    list(range(96)),
    axes[1, 1],
    "Quarter-hour distribution, Porto local time",
    "Quarter-hour of day",
    kind="line",
)
plt.tight_layout()
plt.show()


## Prefix Strategy Decision

Training trajectories are complete, while competition test trajectories are prefixes. The selected strategy samples synthetic train/validation prefix lengths from the unlabeled `test.csv` prefix-length distribution.

In [ ]:
train_profiled = add_polyline_profile(train_raw, require_target=True)
test_profiled = add_polyline_profile(test_raw, require_target=False)

polyline_quality = pd.DataFrame({
    "dataset": ["train_full", "competition_test"],
    "rows": [len(train_profiled), len(test_profiled)],
    "valid_polyline_rows": [int(train_profiled["VALID_POLYLINE"].sum()), int(test_profiled["VALID_POLYLINE"].sum())],
    "invalid_or_too_short_rows": [int((~train_profiled["VALID_POLYLINE"]).sum()), int((~test_profiled["VALID_POLYLINE"]).sum())],
    "missing_data_true_rows": [int(train_profiled["MISSING_DATA"].sum()), int(test_profiled["MISSING_DATA"].sum())],
})
display(polyline_quality)

train_full_lengths = train_profiled.loc[train_profiled["POLYLINE_LENGTH"] >= 2, "POLYLINE_LENGTH"].astype(int)
test_prefix_lengths = test_profiled.loc[test_profiled["POLYLINE_LENGTH"] >= 1, "POLYLINE_LENGTH"].astype(int)
full_minus_destination = (train_full_lengths - 1).rename("full_minus_destination")
uniform_prefix = sample_uniform_prefix_lengths(train_full_lengths, seed=SEED)
test_matched_prefix = sample_prefix_lengths_like_test(train_full_lengths, test_prefix_lengths, seed=SEED, sample_factor=PREFIX_SAMPLE_FACTOR)

prefix_summary = describe_lengths(
    full_minus_destination=full_minus_destination,
    uniform_prefix=uniform_prefix,
    test_matched_prefix=test_matched_prefix,
    competition_test_prefix=test_prefix_lengths,
)
display(prefix_summary)

DISPLAY_MAX_PREFIX_LENGTH = 220
prefix_bins = np.arange(0, DISPLAY_MAX_PREFIX_LENGTH + 5, 5)

def percentage_weights(values):
    """Return histogram weights that make bars sum to 100%."""
    values = pd.Series(values).dropna()
    return np.ones(len(values)) / len(values) * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(test_prefix_lengths, bins=prefix_bins, weights=percentage_weights(test_prefix_lengths), alpha=0.45, label="competition test")
ax.hist(test_matched_prefix, bins=prefix_bins, weights=percentage_weights(test_matched_prefix), alpha=0.45, label="test-matched synthetic")
ax.set_title(f"Selected prefix strategy vs competition test, 0-{DISPLAY_MAX_PREFIX_LENGTH} GPS points")
ax.set_xlabel("Prefix length (GPS points)")
ax.set_ylabel("Rows (%)")
ax.legend()
plt.tight_layout()
plt.show()

## Spatial Filtering Decision

Broad sanity bounds are the selected train-only spatial filter. Porto-core and test-prefix-envelope bounds remain diagnostics, not hard filters.

In [ ]:
train_start = point_frame_from_rows(train_profiled, "train_start", first_point)
train_destination = point_frame_from_rows(train_profiled, "train_destination", last_point)
test_start = point_frame_from_rows(test_profiled, "test_start", first_point)
test_prefix_end = point_frame_from_rows(test_profiled, "test_prefix_end", last_point)
test_prefix_points = flatten_profiled_points(test_profiled, "test_prefix_all_points")

TEST_PREFIX_ENVELOPE_BOUNDS = compute_envelope_bounds(test_prefix_points, margin=TEST_ENVELOPE_MARGIN)
BOUNDS_BY_NAME = {
    "broad_sanity": BROAD_SANITY_BOUNDS,
    "porto_core": PORTO_CORE_BOUNDS,
    "test_prefix_envelope": TEST_PREFIX_ENVELOPE_BOUNDS,
}
BOUND_COLORS = {"broad_sanity": "tab:red", "porto_core": "tab:green", "test_prefix_envelope": "tab:purple"}

display(pd.DataFrame.from_dict(BOUNDS_BY_NAME, orient="index").rename_axis("BOUNDS"))

train_start_for_bounds = train_start.set_index("SOURCE_ROW_ID")
train_destination_for_bounds = train_destination.set_index("SOURCE_ROW_ID")
train_row_ids = train_start_for_bounds.index.union(train_destination_for_bounds.index)

trip_level_rows = []
for bounds_name, bounds in BOUNDS_BY_NAME.items():
    start_inside = bounds_mask(train_start_for_bounds, bounds).reindex(train_row_ids, fill_value=False)
    destination_inside = bounds_mask(train_destination_for_bounds, bounds).reindex(train_row_ids, fill_value=False)
    start_outside = ~start_inside
    destination_outside = ~destination_inside
    any_outside = start_outside | destination_outside
    trip_level_rows.append({
        "BOUNDS": bounds_name,
        "TRAIN_ROWS": len(train_row_ids),
        "START_OUTSIDE_ROWS": int(start_outside.sum()),
        "DESTINATION_OUTSIDE_ROWS": int(destination_outside.sum()),
        "START_OR_DESTINATION_OUTSIDE_ROWS": int(any_outside.sum()),
        "START_OR_DESTINATION_OUTSIDE_PCT": any_outside.mean() * 100 if len(train_row_ids) else np.nan,
    })
trip_level_summary = pd.DataFrame(trip_level_rows).sort_values("BOUNDS")
display(trip_level_summary)

coordinate_points = pd.concat([train_start, train_destination, test_start, test_prefix_end], ignore_index=True)
fig, ax = plt.subplots(figsize=(10, 7))
for point_type, group in coordinate_points.groupby("POINT_TYPE"):
    sample = group.sample(min(len(group), 10_000), random_state=SEED) if len(group) > 10_000 else group
    ax.scatter(sample["LONGITUDE"], sample["LATITUDE"], s=3, alpha=0.30, label=point_type)
for bounds_name, bounds in BOUNDS_BY_NAME.items():
    add_bounds_rectangle(ax, bounds, bounds_name, BOUND_COLORS[bounds_name])
ax.set_title("Spatial start/end points with diagnostic bounds")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.show()

## Duration Artifact Decision

Full trip duration is known only for train rows. The selected rule removes long-duration train trajectories only when the trajectory never went far from its start.

In [ ]:
train_profiled["TRIP_DURATION_MINUTES"] = (
    (train_profiled["POLYLINE_LENGTH"].fillna(0) - 1).clip(lower=0) * SECONDS_PER_GPS_STEP / 60
)
test_profiled["PREFIX_DURATION_MINUTES"] = (
    (test_profiled["POLYLINE_LENGTH"].fillna(0) - 1).clip(lower=0) * SECONDS_PER_GPS_STEP / 60
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
duration_bins = np.arange(0, 41, 1)
axes[0].hist(train_profiled["TRIP_DURATION_MINUTES"].dropna(), bins=duration_bins, alpha=0.75)
axes[0].set_title("Train full trip duration, 0-40 minutes")
axes[0].set_xlabel("Duration (minutes)")
axes[0].set_ylabel("Rows")
axes[0].set_xlim(0, 40)
axes[1].hist(test_profiled["PREFIX_DURATION_MINUTES"].dropna(), bins=duration_bins, alpha=0.75)
axes[1].set_title("Competition test prefix duration, 0-40 minutes")
axes[1].set_xlabel("Duration (minutes)")
axes[1].set_ylabel("Rows")
axes[1].set_xlim(0, 40)
plt.tight_layout()
plt.show()

valid_train_duration = train_profiled.loc[train_profiled["VALID_POLYLINE"]].copy()
valid_train_duration["TRIP_DURATION_MINUTES"] = (
    (valid_train_duration["POLYLINE_LENGTH"] - 1).clip(lower=0) * SECONDS_PER_GPS_STEP / 60
)
valid_train_duration["MAX_DISTANCE_FROM_START_KM"] = valid_train_duration["POLYLINE_POINTS"].map(max_distance_from_start_km)
valid_train_duration["START_LONGITUDE"] = valid_train_duration["POLYLINE_POINTS"].map(lambda points: first_point(points)[0])
valid_train_duration["START_LATITUDE"] = valid_train_duration["POLYLINE_POINTS"].map(lambda points: first_point(points)[1])
valid_train_duration["DESTINATION_LONGITUDE"] = valid_train_duration["POLYLINE_POINTS"].map(lambda points: last_point(points)[0])
valid_train_duration["DESTINATION_LATITUDE"] = valid_train_duration["POLYLINE_POINTS"].map(lambda points: last_point(points)[1])
valid_train_duration["SOURCE_ROW_ID"] = valid_train_duration.index
if "TRIP_ID" not in valid_train_duration.columns:
    valid_train_duration["TRIP_ID"] = valid_train_duration.index

duration_rule_comparison = []
for threshold in [120, 240, 480]:
    long_duration = valid_train_duration["TRIP_DURATION_MINUTES"] > threshold
    local_long = long_duration & (valid_train_duration["MAX_DISTANCE_FROM_START_KM"] <= DURATION_ARTIFACT_MAX_DISTANCE_FROM_START_KM)
    duration_rule_comparison.append({
        "duration_threshold_minutes": threshold,
        "long_duration_rows": int(long_duration.sum()),
        "long_and_max_distance_le_20km_rows": int(local_long.sum()),
        "long_and_max_distance_le_20km_pct": local_long.mean() * 100,
    })
display(pd.DataFrame(duration_rule_comparison))

duration_artifact_candidates = valid_train_duration.loc[
    (valid_train_duration["TRIP_DURATION_MINUTES"] > DURATION_ARTIFACT_MINUTES)
    & (valid_train_duration["MAX_DISTANCE_FROM_START_KM"] <= DURATION_ARTIFACT_MAX_DISTANCE_FROM_START_KM)
].sort_values("TRIP_DURATION_MINUTES", ascending=False)

selected_rule_summary = pd.Series({
    "duration_artifact_minutes": DURATION_ARTIFACT_MINUTES,
    "duration_artifact_max_distance_from_start_km": DURATION_ARTIFACT_MAX_DISTANCE_FROM_START_KM,
    "duration_artifact_candidate_rows": int(len(duration_artifact_candidates)),
    "duration_artifact_candidate_pct": len(duration_artifact_candidates) / len(valid_train_duration) * 100 if len(valid_train_duration) else np.nan,
}, name="selected_duration_artifact_rule")
display(selected_rule_summary)

duration_candidate_columns = [
    "SOURCE_ROW_ID",
    "TRIP_ID",
    "POLYLINE_LENGTH",
    "TRIP_DURATION_MINUTES",
    "MAX_DISTANCE_FROM_START_KM",
    "START_LONGITUDE",
    "START_LATITUDE",
    "DESTINATION_LONGITUDE",
    "DESTINATION_LATITUDE",
]
display(duration_artifact_candidates.head(25)[duration_candidate_columns])

## Feature Scaling And Encoding Notes

GPS coordinates have small local variation but extreme train outliers. `code/build_preprocessed_datasets.py` standardizes extracted GPS columns after synthetic prefix sampling, fitting scaling statistics on synthetic training prefixes only.

In [ ]:
gps_points_for_check = pd.concat([
    train_start[["LONGITUDE", "LATITUDE"]],
    train_destination[["LONGITUDE", "LATITUDE"]],
    test_start[["LONGITUDE", "LATITUDE"]],
    test_prefix_end[["LONGITUDE", "LATITUDE"]],
], ignore_index=True).dropna()

scaling_summary = pd.DataFrame({
    "mean": gps_points_for_check.mean(),
    "std": gps_points_for_check.std(ddof=0),
    "min": gps_points_for_check.min(),
    "max": gps_points_for_check.max(),
})
display(scaling_summary)

## Optional Visual Appendix

GPS density is visual context only. It does not define preprocessing rules. Set `RUN_GPS_DENSITY = True` in setup to render it.

In [ ]:
DENSITY_MAP_BOUNDS = PORTO_CORE_BOUNDS
DENSITY_MAP_BINS = 700


def flatten_points_within_bounds(profiled, bounds):
    """Collect trajectory points inside plotting bounds for density rendering."""
    valid_rows = profiled.loc[profiled["VALID_POLYLINE"], ["POLYLINE_POINTS"]]
    longitudes = []
    latitudes = []
    for points in valid_rows["POLYLINE_POINTS"]:
        for longitude, latitude in points:
            if bounds["lon_min"] <= longitude <= bounds["lon_max"] and bounds["lat_min"] <= latitude <= bounds["lat_max"]:
                longitudes.append(float(longitude))
                latitudes.append(float(latitude))
    return pd.DataFrame({"LONGITUDE": longitudes, "LATITUDE": latitudes}), len(valid_rows)


def set_geographic_axes(axis, bounds):
    """Set geographic limits and approximate aspect ratio for Porto plots."""
    center_latitude = (bounds["lat_min"] + bounds["lat_max"]) / 2
    axis.set_aspect(1 / np.cos(np.radians(center_latitude)))
    axis.set_xlim(bounds["lon_min"], bounds["lon_max"])
    axis.set_ylim(bounds["lat_min"], bounds["lat_max"])
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")


def plot_density_background(axis, density_points, bounds, bins=DENSITY_MAP_BINS):
    """Render a log-scaled GPS density background."""
    axis.set_facecolor("white")
    _, _, _, density_image = axis.hist2d(
        density_points["LONGITUDE"],
        density_points["LATITUDE"],
        bins=bins,
        range=[[bounds["lon_min"], bounds["lon_max"]], [bounds["lat_min"], bounds["lat_max"]]],
        norm=LogNorm(vmin=1),
        cmap="Blues",
        cmin=1,
        alpha=0.78,
    )
    set_geographic_axes(axis, bounds)
    axis.grid(color="#e5e7eb", linewidth=0.5, alpha=0.8)
    axis.set_axisbelow(True)
    return density_image


if RUN_GPS_DENSITY:
    density_points, density_source_rows = flatten_points_within_bounds(train_profiled, DENSITY_MAP_BOUNDS)
    assert len(density_points) > 0, "Density map has no GPS points inside DENSITY_MAP_BOUNDS."
    display(pd.DataFrame([{
        "density_source_train_rows": density_source_rows,
        "density_points_inside_bounds": len(density_points),
        "density_bounds": "PORTO_CORE_BOUNDS",
    }]))

    fig, ax = plt.subplots(figsize=(10, 8))
    density_image = plot_density_background(ax, density_points, DENSITY_MAP_BOUNDS)
    ax.set_title("Porto-area train GPS-density background")
    add_bounds_rectangle(ax, PORTO_CORE_BOUNDS, "porto_core", "tab:green")
    fig.colorbar(density_image, ax=ax, label="Train GPS point density (log scale)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping optional GPS density plot. Set RUN_GPS_DENSITY = True to render it.")

## Takeaways

- Keep test-matched synthetic prefixes; they match the competition test prefix distribution better than full-minus-destination or uniform prefixes.
- Keep Porto-local temporal features and ISO week/weekday/hour/quarter-hour diagnostics; timestamps are valid and rows should not be filtered by date.
- Apply broad spatial sanity bounds as the selected train-only spatial filter; keep Porto-core and test-prefix-envelope only as diagnostics.
- Apply the selected duration artifact rule: drop train rows where `TRIP_DURATION_MINUTES > 240` and `MAX_DISTANCE_FROM_START_KM <= 20`.
- Standardize extracted GPS feature columns in `code/build_preprocessed_datasets.py` using train-fitted statistics.